# SLEAP Import and 3D Triangulation Workflow

Objective: show how SLEAP outputs enter the canonical behavior-lab format and where 2D multi-view tracks connect to 3D triangulation before behavior discovery.

## Why SLEAP And Why Triangulation

SLEAP is a pose estimator and tracker, not a behavior classifier. Its value in this workbench is that it provides time-indexed keypoints and track metadata that can be converted into the canonical `(T, K, D)` format.

Triangulation is the next step when multiple calibrated camera views exist. It turns 2D detections into sparse 3D keypoints, which changes the downstream geometry in a material way: some methods are best on 2D top-view motion, while others assume real 3D coordinates.

In [1]:
from pathlib import Path
import sys
import numpy as np

ROOT = Path.cwd()
if ROOT.name != "behavior-lab":
    ROOT = Path("/Users/joon/dev/behavior-lab")
sys.path.insert(0, str(ROOT / "src"))

from behavior_lab.pose import load_sleap_file
from behavior_lab.data.loaders import get_loader

OUTPUT_DIR = ROOT / "outputs" / "behavior_analysis_workbench"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## Load SLEAP Analysis H5

Set `SLEAP_PATH` to a real `.analysis.h5`, `.h5`, or `.slp`. If absent, this notebook creates a small synthetic SLEAP-style H5 so the import path can be verified.

In [2]:
SLEAP_PATH = None  # Example: ROOT / "data" / "sleap" / "predictions.analysis.h5"

if SLEAP_PATH is None or not Path(SLEAP_PATH).exists():
    try:
        import h5py
    except ImportError:
        h5py = None
    if h5py is not None:
        SLEAP_PATH = OUTPUT_DIR / "synthetic_sleap.analysis.h5"
        tracks = np.zeros((120, 4, 2, 2), dtype="float32")  # frames, nodes, tracks, xy
        t = np.linspace(0, 2 * np.pi, 120)
        tracks[:, :, 0, 0] = np.sin(t)[:, None] + np.arange(4) * 0.1
        tracks[:, :, 0, 1] = np.cos(t)[:, None]
        tracks[:, :, 1, 0] = np.sin(t + 0.5)[:, None] + np.arange(4) * 0.1 + 1.0
        tracks[:, :, 1, 1] = np.cos(t + 0.5)[:, None] + 1.0
        scores = np.ones((120, 4, 2), dtype="float32")
        with h5py.File(SLEAP_PATH, "w") as h5:
            h5.create_dataset("tracks", data=tracks)
            h5.create_dataset("point_scores", data=scores)
            h5.create_dataset("node_names", data=np.array([b"nose", b"neck", b"tail_base", b"tail_tip"]))
            h5.create_dataset("track_names", data=np.array([b"resident", b"intruder"]))

if SLEAP_PATH is not None and Path(SLEAP_PATH).exists():
    result = load_sleap_file(SLEAP_PATH, instance_mode="flatten", confidence_threshold=0.2)
    seq = result.sequences[0]
    print(SLEAP_PATH)
    print(seq.keypoints.shape)
    print(seq.metadata["node_names"], seq.metadata["track_names"])
else:
    seq = None
    print("h5py is not installed and no SLEAP_PATH was provided; install behavior-lab[sleap] to execute this import cell.")

/Users/joon/dev/behavior-lab/outputs/behavior_analysis_workbench/synthetic_sleap.analysis.h5
(120, 8, 2)
['nose', 'neck', 'tail_base', 'tail_tip'] ['resident', 'intruder']


## Loader Factory Path

In [3]:
if SLEAP_PATH is not None and Path(SLEAP_PATH).exists():
    loader = get_loader("sleap", data_dir=Path(SLEAP_PATH).parent, instance_mode="separate")
    sequences = loader.load_all()
    [(s.sample_id, s.keypoints.shape, s.metadata.get("track_name")) for s in sequences]
else:
    []

## Triangulation Handoff

For 3D sparse keypoints, run DLC/SLEAP per camera first, then triangulate with calibrated cameras. This repo already keeps the triangulation script separate from behavior discovery so pose accuracy can be evaluated independently.

In [4]:
TRIANGULATION_COMMAND = """
python scripts/anipose_triangulate.py \
  --predictions-dir outputs/kp_benchmark \
  --calibration data/calibration.toml \
  --out outputs/kp_benchmark/sleap_or_dlc_triangulated_3d.npz
"""
print(TRIANGULATION_COMMAND)


python scripts/anipose_triangulate.py   --predictions-dir outputs/kp_benchmark   --calibration data/calibration.toml   --out outputs/kp_benchmark/sleap_or_dlc_triangulated_3d.npz



## Practical Rule

Keep the import step and the triangulation step separate from discovery. That makes it possible to audit pose noise, missingness, and calibration problems before those errors are mixed into clustering or syllable extraction.

## Downstream Contract

After import or triangulation, downstream modules should receive only canonical keypoints plus metadata. Missing low-confidence points should remain `NaN` until preprocessing decides how to interpolate, mask, or drop them.